# Financial Data Quality Audit

This notebook validates the datasets declared in `reports/data_inventory.yml` and checks if files are present, readable, and structurally consistent with metadata.

## Checks included

1. Inventory schema sanity checks
2. Structural QA (file, sheet, metadata, header/date integrity)
3. ETL smoke test (extract tabular data from each workbook)
4. EDA sanity metrics (rows, date range, duplicates, sorting, missing ratio)
5. Consolidated critical issues vs expected warnings
6. Export of detailed QA tables

In [57]:
from pathlib import Path
import warnings

import pandas as pd
import yaml
from openpyxl import load_workbook

warnings.filterwarnings('ignore', category=UserWarning)

In [58]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

INV_PATH = ROOT / 'reports' / 'data_inventory.yml'
assert INV_PATH.exists(), f'Inventory not found: {INV_PATH}'

with INV_PATH.open('r', encoding='utf-8') as f:
    inv = yaml.safe_load(f)

datasets = inv.get('datasets', [])
df = pd.DataFrame(datasets)
print(f'inventory_file={INV_PATH}')
print(f'datasets={len(df)}')
df.head(3)

inventory_file=C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\data_inventory.yml
datasets=46


,id,path,category,subcategory,ticker,sheet,header_row,data_start_row,freq,variables,unit,status,notes
0,investable_assets_cash_euribor_3m,data/investable_assets/cash/EURIBOR_3M.xlsx,investable_assets,cash,EURIBOR_3M,Sheet 1,10.0,11.0,daily,"[Exchange Date, Fixing Value]",rate_level,OK,
1,investable_assets_commodities_bloomberg_commodity,data/investable_assets/commodities/Bloomberg_C...,investable_assets,commodities,Bloomberg_Commodity,Sheet 1,31.0,32.0,daily,"[Exchange Date, Close, Net, %Chg, Open, Low, H...",price_index_level,OK,
2,investable_assets_commodities_brent,data/investable_assets/commodities/Brent.xlsx,investable_assets,commodities,Brent,Sheet 1,34.0,35.0,daily,"[Exchange Date, Close, Net, %Chg, Open, Low, H...",price_index_level,OK,


In [59]:
required_cols = [
    'id', 'path', 'category', 'subcategory', 'ticker', 'sheet',
    'header_row', 'data_start_row', 'freq', 'variables', 'unit', 'status'
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns in inventory: {missing_cols}')

df['path_obj'] = df['path'].map(lambda p: ROOT / p if isinstance(p, str) else None)
df['file_exists'] = df['path_obj'].map(lambda p: p.exists() if p is not None else False)
df['is_xlsx'] = df['path'].map(lambda p: isinstance(p, str) and p.lower().endswith('.xlsx'))

print('schema_ok=True')
print(df[['status', 'file_exists']].value_counts(dropna=False).rename('count'))

schema_ok=True
status               file_exists
OK                   True           41
EMPTY_METADATA_ONLY  True            2
NEEDS_REFRESH        True            2
WRONG_ENTITY         True            1
Name: count, dtype: int64


In [60]:
def safe_cell_value(ws, row, col=1):
    if row is None or row < 1:
        return None
    return ws.cell(row=row, column=col).value

def normalize_row_index(x):
    if pd.isna(x):
        return None
    try:
        x_num = float(x)
        if x_num.is_integer() and x_num > 0:
            return int(x_num)
    except Exception:
        return None
    return None

def parse_date_safe(x):
    if x is None or x == '':
        return pd.NaT
    if isinstance(x, (int, float)) and not isinstance(x, bool):
        if x > 59:
            dt = pd.to_datetime(x, errors='coerce', unit='D', origin='1899-12-30')
            if pd.notna(dt):
                return dt
    return pd.to_datetime(x, errors='coerce')

def first_numeric_after_date(row_values):
    for v in row_values[1:]:
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            return float(v)
        if isinstance(v, str):
            raw = v.strip().replace('%', '')
            raw = raw.replace(',', '.') if raw.count(',') == 1 and raw.count('.') == 0 else raw.replace(',', '')
            num = pd.to_numeric(raw, errors='coerce')
            if pd.notna(num):
                return float(num)
    return None

def has_live_query_formula(ws, max_rows=40, max_cols=12):
    for r in range(1, min(ws.max_row, max_rows) + 1):
        for c in range(1, max_cols + 1):
            v = ws.cell(r, c).value
            if isinstance(v, str) and v.startswith('=') and ('RDP.' in v or 'TR.' in v or 'HistoricalPricing' in v):
                return True
    return False

def infer_data_start_row(ws, preferred_hint=None, scan_limit=250):
    candidates = []
    if preferred_hint is not None:
        candidates.append(preferred_hint)
    candidates.extend(range(1, min(scan_limit, ws.max_row) + 1))

    seen = set()
    ordered = []
    for c in candidates:
        if c not in seen:
            ordered.append(c)
            seen.add(c)

    for r in ordered:
        raw = safe_cell_value(ws, r, 1)
        dt = parse_date_safe(raw)
        if pd.notna(dt):
            return r
    return None

def extract_timeseries(ws, data_start_hint=None, row_cap=8000):
    dr = infer_data_start_row(ws, preferred_hint=data_start_hint)
    if dr is None:
        return dr, [], []

    dates = []
    values = []
    blank_streak = 0

    for row in ws.iter_rows(min_row=dr, values_only=True):
        raw_date = row[0] if len(row) > 0 else None
        dt = parse_date_safe(raw_date)

        if pd.isna(dt):
            blank_streak += 1
            if blank_streak >= 10 and len(dates) > 0:
                break
            continue

        blank_streak = 0
        dates.append(dt)
        values.append(first_numeric_after_date(row))

        if len(dates) >= row_cap:
            break

    return dr, dates, values

def evaluate_sheet(ws, data_start_hint=None):
    dr_used, dates, values = extract_timeseries(ws, data_start_hint=data_start_hint)
    return {
        'dr_used': dr_used,
        'dates': dates,
        'values': values,
        'n_rows': len(dates),
    }

skip_ids = {'regime_variables_monetary_ecb_balance_sheet'}
skip_statuses = {'WRONG_ENTITY'}
expected_incomplete_statuses = {'EMPTY_METADATA_ONLY', 'NEEDS_REFRESH'}

rows = []
for rec in df.to_dict(orient='records'):
    status = rec.get('status', 'OK')
    if rec.get('id') in skip_ids or status in skip_statuses:
        continue

    hr = normalize_row_index(rec.get('header_row'))
    dr = normalize_row_index(rec.get('data_start_row'))

    out = {
        'id': rec['id'],
        'path': rec['path'],
        'status': status,
        'sheet_expected': rec.get('sheet'),
        'sheet_used': rec.get('sheet'),
        'header_row_expected': hr,
        'data_start_row_expected': dr,
        'data_start_row_used': None,
        'n_variables_expected': len(rec.get('variables') or []),
        'file_exists': False,
        'workbook_ok': False,
        'sheet_ok': False,
        'header_row_has_values': False,
        'n_header_cells_detected': 0,
        'first_date_parse_ok': None,
        'first_date_value': None,
        'n_rows_extracted': 0,
        'n_valid_dates': 0,
        'date_min': None,
        'date_max': None,
        'date_is_monotonic': None,
        'n_duplicate_dates': None,
        'value_missing_ratio': None,
        'issues': [],
        'warnings': []
    }

    p = ROOT / rec['path']
    if not p.exists():
        out['issues'].append('missing_file')
        rows.append(out)
        continue

    out['file_exists'] = True

    try:
        wb = load_workbook(filename=p, data_only=True, read_only=True)
        out['workbook_ok'] = True
    except Exception:
        out['issues'].append('cannot_open_workbook')
        rows.append(out)
        continue

    expected_sheet = rec.get('sheet')
    if expected_sheet not in wb.sheetnames:
        out['issues'].append('missing_sheet')
        rows.append(out)
        continue

    ws_expected = wb[expected_sheet]
    out['sheet_ok'] = True

    if hr is None:
        if status in expected_incomplete_statuses:
            out['warnings'].append('expected_missing_header_row_metadata')
        else:
            out['issues'].append('missing_header_row_metadata')

    if dr is None:
        if status in expected_incomplete_statuses:
            out['warnings'].append('expected_missing_data_start_row_metadata')
        else:
            out['issues'].append('missing_data_start_row_metadata')

    if hr is not None:
        try:
            header_values = list(next(ws_expected.iter_rows(min_row=hr, max_row=hr, values_only=True)))
            non_empty_header = [v for v in header_values if v not in (None, '')]
            out['n_header_cells_detected'] = len(non_empty_header)
            out['header_row_has_values'] = len(non_empty_header) > 0
            if not out['header_row_has_values']:
                if status in expected_incomplete_statuses:
                    out['warnings'].append('expected_empty_header_row')
                else:
                    out['issues'].append('empty_header_row')

            n_expected = out['n_variables_expected']
            if n_expected > 0 and len(non_empty_header) < n_expected:
                out['warnings'].append('header_shorter_than_expected_variables')
        except Exception:
            out['issues'].append('header_read_error')

    extraction = evaluate_sheet(ws_expected, data_start_hint=dr)
    best_sheet_name = expected_sheet
    best_extraction = extraction

    if extraction['n_rows'] == 0:
        fallback_priority = ['First Release Data', 'Table Data', 'VIX_History']
        candidate_sheets = [s for s in fallback_priority if s in wb.sheetnames and s != expected_sheet]
        candidate_sheets.extend([s for s in wb.sheetnames if s not in candidate_sheets and s != expected_sheet])

        for s in candidate_sheets:
            ws_alt = wb[s]
            alt = evaluate_sheet(ws_alt, data_start_hint=None)
            if alt['n_rows'] > best_extraction['n_rows']:
                best_extraction = alt
                best_sheet_name = s

    out['sheet_used'] = best_sheet_name
    out['data_start_row_used'] = best_extraction['dr_used']
    out['n_rows_extracted'] = best_extraction['n_rows']
    out['n_valid_dates'] = best_extraction['n_rows']

    if best_extraction['n_rows'] == 0:
        live_formula = False
        try:
            wb_formula = load_workbook(filename=p, data_only=False, read_only=True)
            if expected_sheet in wb_formula.sheetnames:
                live_formula = has_live_query_formula(wb_formula[expected_sheet])
        except Exception:
            live_formula = False

        if live_formula:
            out['warnings'].append('live_query_formula_without_cached_history')
        elif status in expected_incomplete_statuses:
            out['warnings'].append('expected_no_cached_data_rows')
        else:
            out['issues'].append('no_data_rows_extracted')
    else:
        ds = pd.Series(best_extraction['dates'], dtype='datetime64[ns]')
        out['date_min'] = ds.min()
        out['date_max'] = ds.max()
        monotonic_inc = bool(ds.is_monotonic_increasing)
        monotonic_dec = bool(ds.is_monotonic_decreasing)
        out['date_is_monotonic'] = monotonic_inc or monotonic_dec
        out['n_duplicate_dates'] = int(ds.duplicated().sum())

        val_series = pd.Series(best_extraction['values'], dtype='float64')
        out['value_missing_ratio'] = float(val_series.isna().mean())

        out['first_date_value'] = str(best_extraction['dates'][0])
        out['first_date_parse_ok'] = True

        if out['n_duplicate_dates'] > 0:
            out['warnings'].append('duplicate_dates')
        if not out['date_is_monotonic']:
            out['warnings'].append('date_not_sorted')
        if out['value_missing_ratio'] is not None and out['value_missing_ratio'] > 0.25:
            out['warnings'].append('high_missing_ratio_value_series')
        if best_sheet_name != expected_sheet:
            out['warnings'].append('used_fallback_sheet_for_extraction')

    out['issue_count'] = len(out['issues'])
    out['warning_count'] = len(out['warnings'])
    out['quality_ok'] = out['issue_count'] == 0
    rows.append(out)

qa = pd.DataFrame(rows)
qa = qa.sort_values(['issue_count', 'warning_count', 'id'], ascending=[False, False, True]).reset_index(drop=True)
print(f'skipped_wrong_entity={len(skip_ids)}')
qa.head(8)

skipped_wrong_entity=1


,id,path,status,sheet_expected,sheet_used,header_row_expected,data_start_row_expected,data_start_row_used,n_variables_expected,file_exists,...,date_min,date_max,date_is_monotonic,n_duplicate_dates,value_missing_ratio,issues,warnings,issue_count,warning_count,quality_ok
0,regime_variables_inflation_eurozone_ppi,data/regime_variables/inflation/Eurozone_PPI.xlsx,EMPTY_METADATA_ONLY,Historical Values,Historical Values,NaN,NaN,NaN,0,True,...,NaT,NaT,None,NaN,NaN,[],"[expected_missing_header_row_metadata, expecte...",0,3,True
1,regime_variables_monetary_germany_2y_yield,data/regime_variables/monetary/Germany_2Y_Yiel...,EMPTY_METADATA_ONLY,Historical Values,Historical Values,NaN,NaN,NaN,0,True,...,NaT,NaT,None,NaN,NaN,[],"[expected_missing_header_row_metadata, expecte...",0,3,True
2,regime_variables_inflation_germany_hicp,data/regime_variables/inflation/Germany_HICP.xlsx,NEEDS_REFRESH,Historical Values,First Release Data,20.0,21.0,5.0,8,True,...,1996-02-01,2026-04-01,True,0.0,0.267218,[],"[high_missing_ratio_value_series, used_fallbac...",0,2,True
3,regime_variables_sentiment_zew_germany,data/regime_variables/sentiment/ZEW_Germany.xlsx,NEEDS_REFRESH,Historical Values,First Release Data,19.0,20.0,5.0,5,True,...,1991-12-01,2026-04-01,True,0.0,0.360775,[],"[high_missing_ratio_value_series, used_fallbac...",0,2,True
4,investable_assets_cash_euribor_3m,data/investable_assets/cash/EURIBOR_3M.xlsx,OK,Sheet 1,Sheet 1,10.0,11.0,11.0,2,True,...,2000-01-03,2026-03-31,True,0.0,0.000000,[],[],0,0,True
5,investable_assets_commodities_bloomberg_commodity,data/investable_assets/commodities/Bloomberg_C...,OK,Sheet 1,Sheet 1,31.0,32.0,32.0,8,True,...,2000-01-03,2026-03-31,True,0.0,0.000000,[],[],0,0,True
6,investable_assets_commodities_brent,data/investable_assets/commodities/Brent.xlsx,OK,Sheet 1,Sheet 1,34.0,35.0,35.0,11,True,...,2000-01-04,2026-03-31,True,0.0,0.000741,[],[],0,0,True
7,investable_assets_commodities_gold,data/investable_assets/commodities/Gold.xlsx,OK,Table Data,Table Data,2.0,3.0,3.0,2,True,...,2000-01-03,2026-03-31,True,0.0,0.000000,[],[],0,0,True


In [61]:
total = len(qa)
ok = int(qa['quality_ok'].sum())
fail = total - ok
score = round(100 * ok / total, 2) if total else 0.0

print(f'total_datasets={total}')
print(f'critical_ok={ok}')
print(f'critical_fail={fail}')
print(f'critical_score_pct={score}')
print(f'total_warnings={int(qa["warning_count"].sum())}')

issue_table = (
    qa.explode('issues')
      .dropna(subset=['issues'])
      .groupby('issues', as_index=False)
      .size()
      .sort_values('size', ascending=False)
)

warning_table = (
    qa.explode('warnings')
      .dropna(subset=['warnings'])
      .groupby('warnings', as_index=False)
      .size()
      .sort_values('size', ascending=False)
)

print('\nCritical issues:')
display(issue_table if len(issue_table) else pd.DataFrame({'issues': [], 'size': []}))
print('\nWarnings:')
display(warning_table if len(warning_table) else pd.DataFrame({'warnings': [], 'size': []}))

total_datasets=45
critical_ok=45
critical_fail=0
critical_score_pct=100.0
total_warnings=10

Critical issues:


,issues,size



Warnings:


,warnings,size
0,expected_missing_data_start_row_metadata,2
1,expected_missing_header_row_metadata,2
2,high_missing_ratio_value_series,2
3,live_query_formula_without_cached_history,2
4,used_fallback_sheet_for_extraction,2


In [62]:
problematic = qa.loc[(qa['issue_count'] > 0) | (qa['warning_count'] > 0), [
    'id', 'path', 'status', 'sheet_expected', 'header_row_expected',
    'data_start_row_expected', 'n_rows_extracted', 'date_min', 'date_max',
    'n_duplicate_dates', 'value_missing_ratio', 'issue_count', 'warning_count', 'issues', 'warnings'
]]
problematic = problematic.sort_values(['issue_count', 'warning_count', 'status', 'id'], ascending=[False, False, True, True])
problematic

,id,path,status,sheet_expected,header_row_expected,data_start_row_expected,n_rows_extracted,date_min,date_max,n_duplicate_dates,value_missing_ratio,issue_count,warning_count,issues,warnings
0,regime_variables_inflation_eurozone_ppi,data/regime_variables/inflation/Eurozone_PPI.xlsx,EMPTY_METADATA_ONLY,Historical Values,NaN,NaN,0,NaT,NaT,NaN,NaN,0,3,[],"[expected_missing_header_row_metadata, expecte..."
1,regime_variables_monetary_germany_2y_yield,data/regime_variables/monetary/Germany_2Y_Yiel...,EMPTY_METADATA_ONLY,Historical Values,NaN,NaN,0,NaT,NaT,NaN,NaN,0,3,[],"[expected_missing_header_row_metadata, expecte..."
2,regime_variables_inflation_germany_hicp,data/regime_variables/inflation/Germany_HICP.xlsx,NEEDS_REFRESH,Historical Values,20.0,21.0,363,1996-02-01,2026-04-01,0.0,0.267218,0,2,[],"[high_missing_ratio_value_series, used_fallbac..."
3,regime_variables_sentiment_zew_germany,data/regime_variables/sentiment/ZEW_Germany.xlsx,NEEDS_REFRESH,Historical Values,19.0,20.0,413,1991-12-01,2026-04-01,0.0,0.360775,0,2,[],"[high_missing_ratio_value_series, used_fallbac..."


In [63]:
def suggest_transform(unit, freq, missing_ratio):
    unit = '' if pd.isna(unit) else str(unit).lower()
    freq = '' if pd.isna(freq) else str(freq).lower()

    if 'yield' in unit or 'rate' in unit:
        primary = 'level_or_diff_1'
    elif 'volatility' in unit:
        primary = 'log_or_level'
    elif 'price' in unit or 'index' in unit or 'fx' in unit:
        primary = 'log_then_diff_1'
    elif 'sentiment' in unit or 'diffusion' in unit:
        primary = 'level_or_zscore'
    else:
        primary = 'level_then_stationarity_test'

    seasonality = 'seasonal_check' if freq in {'monthly', 'quarterly'} else 'no_seasonal_adjustment_needed'
    missing_policy = 'impute_short_gaps_only' if (pd.notna(missing_ratio) and missing_ratio <= 0.05) else 'no_blind_imputation'

    return primary, seasonality, missing_policy

if 'research_readiness' not in globals():
    print('Run the readiness scoring cell first to build `research_readiness`.')
else:
    transform_plan = research_readiness[['id', 'freq', 'unit', 'value_missing_ratio', 'paper_ready', 'grade']].copy()
    transform_plan[['primary_transform', 'seasonality_policy', 'missing_data_policy']] = transform_plan.apply(
        lambda r: pd.Series(suggest_transform(r['unit'], r['freq'], r['value_missing_ratio'])),
        axis=1
    )

    transform_plan['pre_model_checks'] = transform_plan.apply(
        lambda r: 'adf_kpss, outlier_scan, break_test' if r['paper_ready'] else 'fix_quality_first_then_run_tests',
        axis=1
    )

    transform_plan = transform_plan.sort_values(['paper_ready', 'grade', 'id'], ascending=[False, True, True])
    transform_plan.head(12)

In [64]:
freq_min_obs = {
    'daily': 252 * 3,      # at least 3 years of trading days
    'monthly': 12 * 8,     # at least 8 years
    'quarterly': 4 * 10,   # at least 10 years
    None: 24,
}

def min_obs_for_freq(freq):
    if not isinstance(freq, str):
        return freq_min_obs[None]
    return freq_min_obs.get(freq.lower(), 24)

rq = qa.merge(
    df[['id', 'category', 'subcategory', 'ticker', 'freq', 'unit']],
    on='id',
    how='left'
).copy()

rq['min_obs_required'] = rq['freq'].map(min_obs_for_freq)
rq['coverage_ratio'] = (rq['n_rows_extracted'] / rq['min_obs_required']).clip(upper=2.0)
rq['coverage_ok'] = rq['n_rows_extracted'] >= rq['min_obs_required']

# Weighted scoring: critical errors dominate, warnings penalize softly.
base_score = 100
rq['score'] = (
    base_score
    - rq['issue_count'] * 40
    - rq['warning_count'] * 6
    - (~rq['coverage_ok']).astype(int) * 12
    - (rq['value_missing_ratio'].fillna(0) > 0.25).astype(int) * 6
).clip(lower=0, upper=100)

def grade_from_score(x):
    if x >= 90:
        return 'A'
    if x >= 80:
        return 'B'
    if x >= 70:
        return 'C'
    return 'D'

rq['grade'] = rq['score'].map(grade_from_score)
rq['paper_ready'] = (rq['issue_count'] == 0) & (rq['grade'].isin(['A', 'B']))

readiness_cols = [
    'id', 'category', 'subcategory', 'ticker', 'freq', 'unit',
    'n_rows_extracted', 'min_obs_required', 'coverage_ratio', 'coverage_ok',
    'issue_count', 'warning_count', 'value_missing_ratio', 'score', 'grade', 'paper_ready',
    'sheet_expected', 'sheet_used', 'issues', 'warnings',
]
research_readiness = rq[readiness_cols].sort_values(['paper_ready', 'score', 'id'], ascending=[False, False, True])

print(f"paper_ready={int(research_readiness['paper_ready'].sum())}/{len(research_readiness)}")
print('grade_distribution:')
print(research_readiness['grade'].value_counts().sort_index())
research_readiness.head(12)

paper_ready=43/45
grade_distribution:
grade
A    40
B     3
C     2
Name: count, dtype: int64


,id,category,subcategory,ticker,freq,unit,n_rows_extracted,min_obs_required,coverage_ratio,coverage_ok,issue_count,warning_count,value_missing_ratio,score,grade,paper_ready,sheet_expected,sheet_used,issues,warnings
4,investable_assets_cash_euribor_3m,investable_assets,cash,EURIBOR_3M,daily,rate_level,6717,756,2.0,True,0,0,0.000000,100,A,True,Sheet 1,Sheet 1,[],[]
5,investable_assets_commodities_bloomberg_commodity,investable_assets,commodities,Bloomberg_Commodity,daily,price_index_level,6646,756,2.0,True,0,0,0.000000,100,A,True,Sheet 1,Sheet 1,[],[]
6,investable_assets_commodities_brent,investable_assets,commodities,Brent,daily,price_index_level,6748,756,2.0,True,0,0,0.000741,100,A,True,Sheet 1,Sheet 1,[],[]
7,investable_assets_commodities_gold,investable_assets,commodities,Gold,daily,price_index_level,6825,756,2.0,True,0,0,0.000000,100,A,True,Table Data,Table Data,[],[]
8,investable_assets_equity_cac40,investable_assets,equity,CAC 40,daily,price_index_level,6714,756,2.0,True,0,0,0.000000,100,A,True,Sheet 1,Sheet 1,[],[]
9,investable_assets_equity_dax,investable_assets,equity,DAX,daily,price_index_level,6667,756,2.0,True,0,0,0.000000,100,A,True,Sheet 1,Sheet 1,[],[]
10,investable_assets_equity_eurostoxx50,investable_assets,equity,EuroStoxx50,daily,price_index_level,6758,756,2.0,True,0,0,0.003107,100,A,True,Sheet 1,Sheet 1,[],[]
11,investable_assets_equity_ftse_mib,investable_assets,equity,FTSE_MIB,daily,price_index_level,6678,756,2.0,True,0,0,0.000000,100,A,True,Sheet 1,Sheet 1,[],[]
12,investable_assets_equity_ibex35,investable_assets,equity,IBEX35,daily,price_index_level,6685,756,2.0,True,0,0,0.000150,100,A,True,Sheet 1,Sheet 1,[],[]
13,investable_assets_equity_stoxxeurope600,investable_assets,equity,StoxxEurope600,daily,price_index_level,6754,756,2.0,True,0,0,0.001925,100,A,True,Sheet 1,Sheet 1,[],[]


## Research-grade readiness for Q1 paper

This section converts QA outputs into a publication-oriented readiness framework:

1. Coverage adequacy by frequency (minimum expected history).
2. Hard/soft penalties from issues and warnings.
3. Final grade per dataset and aggregate dashboard.
4. Suggested transformation pipeline for econometric modeling.

In [65]:
OUT_DIR = ROOT / 'reports' / 'quality'
OUT_DIR.mkdir(parents=True, exist_ok=True)

detailed_path = OUT_DIR / 'data_quality_detailed.csv'
issues_path = OUT_DIR / 'data_quality_issues.csv'
summary_path = OUT_DIR / 'data_quality_issue_summary.csv'
warning_summary_path = OUT_DIR / 'data_quality_warning_summary.csv'

qa.to_csv(detailed_path, index=False)
problematic.to_csv(issues_path, index=False)
issue_table.to_csv(summary_path, index=False)
warning_table.to_csv(warning_summary_path, index=False)

print(f'written: {detailed_path}')
print(f'written: {issues_path}')
print(f'written: {summary_path}')
print(f'written: {warning_summary_path}')

written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_detailed.csv
written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_issues.csv
written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_issue_summary.csv
written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_warning_summary.csv


In [66]:
if 'q1' not in globals() or 'q1_tier_summary' not in globals():
    print('Run the Q1 readiness diagnostics cell first to build `q1` and `q1_tier_summary`.')
else:
    q1_path = OUT_DIR / 'data_quality_q1_readiness.csv'
    q1_summary_path = OUT_DIR / 'data_quality_q1_readiness_summary.csv'

    q1.to_csv(q1_path, index=False)
    q1_tier_summary.to_csv(q1_summary_path, index=False)

    priority = q1.loc[q1['readiness_tier'].isin(['BLOCKER', 'REFRESH_REQUIRED', 'REVIEW']), [
        'id', 'readiness_tier', 'freq_declared', 'n_rows_extracted',
        'cadence_ok', 'last_obs_age_days', 'outlier_rate_robust', 'sheet_used'
    ]].sort_values(['readiness_tier', 'last_obs_age_days'], ascending=[True, False])

    print(f'written: {q1_path}')
    print(f'written: {q1_summary_path}')
    print('\nPriority queue for data engineering:')
    priority.head(20)

written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_q1_readiness.csv
written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_q1_readiness_summary.csv

Priority queue for data engineering:


In [67]:
import numpy as np

def expected_step_bounds(freq):
    # Wide but meaningful tolerances for financial/economic calendars.
    bounds = {
        'daily': (1, 7),
        'weekly': (5, 10),
        'monthly': (25, 35),
        'quarterly': (80, 100),
        'yearly': (330, 390),
    }
    return bounds.get(str(freq).lower(), (None, None))

def robust_outlier_rate(values):
    s = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if len(s) < 20:
        return np.nan
    med = s.median()
    mad = (s - med).abs().median()
    if mad == 0 or pd.isna(mad):
        return np.nan
    robust_z = 0.6745 * (s - med) / mad
    return float((robust_z.abs() > 5).mean())

def extract_dates_values_from_row(row):
    p = ROOT / row['path']
    if not p.exists():
        return [], []

    try:
        wb = load_workbook(filename=p, data_only=True, read_only=True)
    except Exception:
        return [], []

    sheet = row.get('sheet_used') or row.get('sheet_expected')
    if sheet not in wb.sheetnames:
        return [], []

    ws = wb[sheet]
    hint = normalize_row_index(row.get('data_start_row_used'))
    if hint is None:
        hint = normalize_row_index(row.get('data_start_row_expected'))

    _, dates, values = extract_timeseries(ws, data_start_hint=hint, row_cap=10000)
    return dates, values

readiness_rows = []
today = pd.Timestamp.today().normalize()

for _, r in qa.iterrows():
    rec = r.to_dict()
    dates, values = extract_dates_values_from_row(rec)

    freq_declared = df.loc[df['id'].eq(rec['id']), 'freq'].iloc[0] if rec['id'] in set(df['id']) else None
    lo, hi = expected_step_bounds(freq_declared)

    date_min = pd.NaT
    date_max = pd.NaT
    median_step_days = np.nan
    cadence_ok = None
    coverage_years = np.nan
    last_obs_age_days = np.nan
    stale_flag = False

    if len(dates) >= 2:
        ds = pd.Series(pd.to_datetime(dates, errors='coerce')).dropna().sort_values().drop_duplicates()
        if len(ds) >= 2:
            steps = ds.diff().dropna().dt.days
            if len(steps) > 0:
                median_step_days = float(steps.median())
                if lo is not None and hi is not None:
                    cadence_ok = bool(lo <= median_step_days <= hi)

            date_min = ds.min()
            date_max = ds.max()
            coverage_years = float((date_max - date_min).days / 365.25)
            last_obs_age_days = float((today - date_max.normalize()).days)

            stale_threshold = {
                'daily': 14,
                'weekly': 21,
                'monthly': 75,
                'quarterly': 140,
                'yearly': 420,
            }.get(str(freq_declared).lower(), 90)
            stale_flag = bool(last_obs_age_days > stale_threshold)

    outlier_rate = robust_outlier_rate(values)

    warning_list = list(rec.get('warnings', [])) if isinstance(rec.get('warnings', []), list) else []
    has_live_formula = 'live_query_formula_without_cached_history' in warning_list

    if rec.get('issue_count', 0) > 0:
        tier = 'BLOCKER'
    elif has_live_formula:
        tier = 'REFRESH_REQUIRED'
    elif (cadence_ok is False) or stale_flag or (pd.notna(outlier_rate) and outlier_rate > 0.08):
        tier = 'REVIEW'
    else:
        tier = 'READY'

    readiness_rows.append({
        'id': rec['id'],
        'status': rec.get('status'),
        'path': rec.get('path'),
        'sheet_used': rec.get('sheet_used'),
        'freq_declared': freq_declared,
        'n_rows_extracted': rec.get('n_rows_extracted'),
        'issue_count': rec.get('issue_count'),
        'warning_count': rec.get('warning_count'),
        'median_step_days': median_step_days,
        'cadence_ok': cadence_ok,
        'coverage_years': coverage_years,
        'last_obs_age_days': last_obs_age_days,
        'stale_flag': stale_flag,
        'outlier_rate_robust': outlier_rate,
        'readiness_tier': tier,
    })

q1 = pd.DataFrame(readiness_rows).sort_values(['readiness_tier', 'issue_count', 'warning_count', 'id'])
q1_tier_summary = q1.groupby('readiness_tier', as_index=False).size().sort_values('size', ascending=False)

display(q1_tier_summary)
q1.head(15)

,readiness_tier,size
0,READY,23
2,REVIEW,20
1,REFRESH_REQUIRED,2


,id,status,path,sheet_used,freq_declared,n_rows_extracted,issue_count,warning_count,median_step_days,cadence_ok,coverage_years,last_obs_age_days,stale_flag,outlier_rate_robust,readiness_tier
17,regime_variables_global_world_manufacturing_pmi,OK,data/regime_variables/global/World_Manufacturi...,Historical Values,monthly,36,0,0,31.0,True,2.918549,24.0,False,0.000000,READY
18,regime_variables_growth_eurozona_unemployment,OK,data/regime_variables/growth/Eurozona_Unemploy...,Historical Values,monthly,265,0,0,31.0,True,21.998631,24.0,False,0.000000,READY
19,regime_variables_growth_eurozone_gdp_revised_qq,OK,data/regime_variables/growth/Eurozone_GDP_Revi...,Historical Values,quarterly,89,0,0,91.5,True,21.998631,24.0,False,0.067416,READY
20,regime_variables_growth_eurozone_industrial_pr...,OK,data/regime_variables/growth/Eurozone_Industri...,Historical Values,monthly,265,0,0,31.0,True,21.998631,55.0,False,0.015094,READY
21,regime_variables_growth_eurozone_pmi_manufacto...,OK,data/regime_variables/growth/Eurozone_PMI_Manu...,Historical Values,monthly,255,0,0,31.0,True,21.245722,24.0,False,0.000000,READY
22,regime_variables_growth_eurozone_pmi_services,OK,data/regime_variables/growth/Eurozone_PMI_Serv...,Historical Values,monthly,257,0,0,31.0,True,21.412731,-6.0,False,0.007782,READY
23,regime_variables_growth_germany_gdp_detailed_qq,OK,data/regime_variables/growth/Germany_GDP_Detai...,Historical Values,quarterly,89,0,0,91.5,True,21.998631,24.0,False,0.044944,READY
24,regime_variables_growth_spain_gdp_final_qq,OK,data/regime_variables/growth/Spain_GDP_Final_Q...,Historical Values,quarterly,87,0,0,91.5,True,21.497604,24.0,False,0.045977,READY
25,regime_variables_inflation_eurozone_hicp,OK,data/regime_variables/inflation/Eurozone_HICP....,Historical Values,monthly,264,0,0,31.0,True,21.916496,24.0,False,0.011364,READY
26,regime_variables_inflation_eurozone_hicp_core,OK,data/regime_variables/inflation/Eurozone_HICP_...,Historical Values,monthly,363,0,0,31.0,True,30.162902,24.0,False,0.000000,READY


## Q1 Research Readiness Layer

This section adds paper-grade diagnostics beyond structural QA:

1. Cadence consistency vs declared frequency
2. Data staleness (last observation age)
3. Robust outlier intensity (MAD-based)
4. Final readiness tier with concrete actions

## Interpretation guide

- `issues`: critical blockers for production modeling (must fix).
- `warnings`: non-blocking signals to review.
- `live_query_formula_without_cached_history`: the workbook contains live RDP/TR formula queries, but cached historical rows are not saved in the file.
- `used_fallback_sheet_for_extraction`: extraction used an alternative sheet (for example `First Release Data`) because expected sheet had no cached rows.
- `expected_missing_*`: metadata intentionally incomplete for `EMPTY_METADATA_ONLY` sources.
- `high_missing_ratio_value_series`: extracted numeric series has many missing values; check if the selected value column is the right target for your model.

Use this notebook as ETL + EDA pre-flight validation before running empirical models.